![TrainingNotebookLogo.png](https://downloads.limelightvision.io/content/TrainingNotebookLogo.png)

To train a neural object detector for Limelight, click the "play" button on each code block. Pay extra attention to any "❗" you see. By the end of this tutorial, you will have downloaded a .zip file containing your model and label files.

See https://docs.limelightvision.io/docs/docs-limelight/pipeline-neural/training-your-own-detector for a more in-depth tutorial.

In [1]:
# Colab: conflict-free TensorFlow 2.19 stack (Python 3.11)
# Strategy: remove troublemakers, then pin versions that agree with TF 2.19.

%pip -q install --upgrade pip setuptools wheel

# 1) Remove packages that force incompatible NumPy/protobuf ranges
#    (safe even if not installed).
%pip -q uninstall -y tensorflow-model-optimization numba numpy || true

# 2) Satisfy ecosystem constraints FIRST
#    - TF 2.19: NumPy >=1.26,<2.2  → choose 2.1.x (also OK for JAX/OpenCV/etc.)
#    - Many libs (ydf, grpcio-status) want protobuf >=5.x,<6
#    - IPython 7.x nags for jedi
%pip -q install "numpy>=2.1,<2.2" "protobuf>=5.29,<6" jedi

# 3) (Optional) If you use RAPIDS/umap/shap/librosa, they expect numba.
#    Pick a version compatible with NumPy 2.1.x and RAPIDS caps.
%pip -q install "numba>=0.61,<0.62"

# 4) Now install TensorFlow + TF Models + pycocotools
#    Note: 2.19.0 of tf-models-official is yanked but still works;
#    if it ever fails, change it to 2.18.0 (works fine with TF 2.19 for most tasks).
%pip -q install "tensorflow==2.19.0" "tf-models-official==2.19.0" "pycocotools==2.0.7"

import tensorflow as tf, numpy as np, google.protobuf
print("TF:", tf.__version__, "| NumPy:", np.__version__, "| Protobuf:", google.protobuf.__version__)


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
thinc 8.3.6 requires numpy<3.0.0,>=2.0.0, but you have numpy 1.26.4 which is incompatible.
pytensor 2.35.1 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 25.6.0 requires numba<0.62.0a0,>=0.59.1, which is not installed.
cudf-cu12 25.6.0 requires numba<0.62.0a0,>=0.59.1, which is not installed.
stumpy 1.13.0 requires numba>=0.57.1, which is not installed.
cuml-cu12 25.6.0 requires numba<0.62.0a0,>=0.59.1, which is not installed.
umap-learn 0.5.9.post2 requires numba>=0.51.2, which is not installed.
shap 0.49.1 requires numba>=0.54, which is not installe

In [6]:
from google.colab import files
up = files.upload()  # select your TFRecord .zip from Roboflow
zip_name = list(up.keys())[0]
!unzip -o "$zip_name" -d /content/data
!find /content/data -maxdepth 2 -type f -name "*.record" -o -name "label_map.pbtxt"


Saving Decode.v2i.tfrecord.zip to Decode.v2i.tfrecord.zip
Archive:  Decode.v2i.tfrecord.zip
  inflating: /content/data/README.dataset.txt  
  inflating: /content/data/README.roboflow.txt  
   creating: /content/data/test/
 extracting: /content/data/test/objects.tfrecord  
  inflating: /content/data/test/objects_label_map.pbtxt  
   creating: /content/data/train/
 extracting: /content/data/train/objects.tfrecord  
  inflating: /content/data/train/objects_label_map.pbtxt  
   creating: /content/data/valid/
 extracting: /content/data/valid/objects.tfrecord  
  inflating: /content/data/valid/objects_label_map.pbtxt  


In [16]:
# --- Install TF Object Detection API *as a package* + compile protos, fix PYTHONPATH ---
import sys, os, pathlib

# 1) Get the repo (ok if it already exists)
if not pathlib.Path("/content/models").exists():
    !git clone https://github.com/tensorflow/models.git /content/models

# 2) Install the packaged OD-API from its setup (use a stable tag)
#    NOTE: this installs the 'object-detection' python package properly.
%pip -q install "git+https://github.com/tensorflow/models.git@v2.13.0#egg=object_detection&subdirectory=research/object_detection/packages/tf2"

# 3) Ensure protoc is installed, then compile protos (needed for runtime)
!sudo apt -y -qq install protobuf-compiler python3-dev > /dev/null
%cd /content/models/research
!protoc object_detection/protos/*.proto --python_out=.

# 4) Back to root; set both PYTHONPATH and sys.path explicitly
%cd /content
os.environ["PYTHONPATH"] = "/content:/content/models:/content/models/research:/content/models/research/slim:" + os.environ.get("PYTHONPATH","")
sys.path = os.environ["PYTHONPATH"].split(":") + sys.path

# 5) Sanity check imports
from object_detection.utils import config_util, label_map_util
import object_detection
print("✅ OD-API installed & importable")


ERROR: object_detection from git+https://github.com/tensorflow/models.git@v2.13.0#egg=object_detection&subdirectory=research/object_detection/packages/tf2 does not appear to be a Python project: neither 'setup.py' nor 'pyproject.toml' found.


/content/models/research
/content


ModuleNotFoundError: No module named 'object_detection'

In [8]:
# What files did we actually get?
!echo "=== TREE /content/data ==="
!find /content/data -maxdepth 3 -type f | sed 's|/content/data/||' | sort | head -n 200

# If you downloaded via curl and named it decode_dataset.zip, list its contents too:
import glob, os
zips = glob.glob("*.zip")
print("\n=== ZIP files present in /content ===", zips)
if zips:
    import zipfile
    for z in zips:
        print(f"\n--- Listing first 60 entries of {z} ---")
        with zipfile.ZipFile(z) as f:
            for i, n in enumerate(f.namelist()):
                print(n)
                if i>60: break


=== TREE /content/data ===
README.dataset.txt
README.roboflow.txt
test/objects_label_map.pbtxt
test/objects.tfrecord
train/objects_label_map.pbtxt
train/objects.tfrecord
valid/objects_label_map.pbtxt
valid/objects.tfrecord

=== ZIP files present in /content === ['Decode.v2i.tfrecord.zip']

--- Listing first 60 entries of Decode.v2i.tfrecord.zip ---
README.dataset.txt
README.roboflow.txt
test/
test/objects.tfrecord
test/objects_label_map.pbtxt
train/
train/objects.tfrecord
train/objects_label_map.pbtxt
valid/
valid/objects.tfrecord
valid/objects_label_map.pbtxt


# 1. Install The Object Detection Package

Test the environment by running `model_builder_tf2_test.py` to make sure everything is working as expected.

# 1.1. Get Dataset From Google Drive

1. Expand this section
2. Upload your RoboFlow .tfrecord.zip to Google Drive
3. Share the uploaded .tfrecord.zip such that anyone with the link can access the file.
4. Run this block
5. Paste your Google Drive file share link into the text box that appears after running this block
6. Click the "Process Dataset" Buttton
7. Click the Refresh button in the "Files" pane to ensure dataset.zip exists

# 2. Auto-detect relevant tfrecord components

In [ ]:
datasetPath = '/content/dataset.zip'
print(datasetPath)
!unzip $datasetPath

In [ ]:
import os
import fnmatch

def find_files(directory, pattern):
    for root, dirs, files in os.walk(directory):
        for basename in files:
            if fnmatch.fnmatch(basename, pattern):
                filename = os.path.join(root, basename)
                yield filename

def set_tfrecord_variables(directory):
    train_record_fname = ''
    val_record_fname = ''
    label_map_pbtxt_fname = ''

    for tfrecord_file in find_files(directory, '*.tfrecord'):
        if '/train/' in tfrecord_file:
            train_record_fname = tfrecord_file
        elif '/valid/' in tfrecord_file:
            val_record_fname = tfrecord_file
        elif '/test/' in tfrecord_file:
            pass

    for label_map_file in find_files(directory, '*_label_map.pbtxt'):
        label_map_pbtxt_fname = label_map_file  # Assuming one common label map file

    return train_record_fname, val_record_fname, label_map_pbtxt_fname


train_record_fname, val_record_fname, label_map_pbtxt_fname = set_tfrecord_variables('/content')

#if(MLENVIRONMENT=="COLAB"):
    #train_record_fname = '/content/train/cubes-cones.tfrecord'
    #val_record_fname = '/content/valid/cubes-cones.tfrecord'
    #label_map_pbtxt_fname = '/content/train/cubes-cones_label_map.pbtxt'

print("Train Record File:", train_record_fname)
print("Validation Record File:", val_record_fname)
print("Label Map File:", label_map_pbtxt_fname)



# 3.&nbsp;Training Configuration and Labels File Generation

Download the pre-trained Limelight Base Model

Generate Labels File

# 4.&nbsp;Train Model

Once training starts, come back and click the refresh button within the tensorboard window to check training progress.



Fix TF 2.15 breaking changes

Train

Feel free to stop training early. Check the 'training_progress' folder to see all training checkpoints.


# 5.&nbsp;Convert Model to TFLite

# 6. Quantize model
The "TFLiteConverter" module will perform [post-training quantization](https://www.tensorflow.org/lite/performance/post_training_quantization) on the model. To quantize the model, we need to provide a set of example images. We will extract 100 images from the training tfrecord and place said images into the "extracted_samples" folder.


In [ ]:
# A generator that provides a representative dataset
# Code modified from https://colab.research.google.com/github/google-coral/tutorials/blob/master/retrain_classification_ptq_tf2.ipynb

# First, get input details for model so we know how to preprocess images
interpreter = tf.lite.Interpreter(model_path=model_path_32bit)
interpreter.allocate_tensors()
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()
height = input_details[0]['shape'][1]
width = input_details[0]['shape'][2]

import random

def representative_data_gen():
  dataset_list = quant_image_list
  quant_num = 300
  for i in range(quant_num):
    pick_me = random.choice(dataset_list)
    print(pick_me)
    image = tf.io.read_file(pick_me)

    if pick_me.endswith('.jpg') or pick_me.endswith('.JPG') or pick_me.endswith('.jpeg'):
      image = tf.io.decode_jpeg(image, channels=3)
    elif pick_me.endswith('.png'):
      image = tf.io.decode_png(image, channels=3)
    elif pick_me.endswith('.bmp'):
      image = tf.io.decode_bmp(image, channels=3)

    image = tf.image.resize(image, [width, height])  # TO DO: Replace 300s with an automatic way of reading network input size
    image = tf.cast(image / 255., tf.float32)
    image = tf.expand_dims(image, 0)
    yield [image]

Finally, we'll initialize the TFLiteConverter module, point it at the TFLite graph we generated in Step 6, and provide it with the representative dataset generator function we created in the previous code block. We'll configure the converter to quantize the model's weight values to INT8 format.

# 7. Compile Model for Limelight & Download


Install Coral Compiler

Compile the previously-generated 8-bit model for Google Coral

In [ ]:
!cd {FINALOUTPUTFOLDER} && pwd && edgetpu_compiler limelight_neural_detector_8bit.tflite && pwd && mv limelight_neural_detector_8bit_edgetpu.tflite limelight_neural_detector_coral.tflite && rm limelight_neural_detector_8bit_edgetpu.log

Zip models

In [ ]:
!rm {HOMEFOLDER}limelight_detectors.zip
!zip -r {HOMEFOLDER}limelight_detectors.zip {FINALOUTPUTFOLDER}

Download

In [ ]:
from google.colab import files
files.download(HOMEFOLDER+'limelight_detectors.zip')